# Carnage v1 on Colab

1v1 or 2v2 Rocket League trainer (GigaLearnCPP + RLGymCPP), **Nexto-style flat obs** (1v1 = 94 dims, 2v2 = 128 dims), Phase 1 config.

**Config (Phase 1):** Policy/Critic [1024,1024,512,512] no shared head | 3 epochs, ts/itr 50k, batch 50k, minibatch 25k, LR 2e-4, entropy 0.05 | Rewards: Touch(50), SpeedTowardBall(5), FaceBall(1), Air(0.15) | Terminal: no-touch 10s or goal scored | RandomState.
**Team modes (2v2):** each reward is wrapped in team spirit (ZealanL guide) so teammates share it - start low (0.1) and raise towards 1.0 as the bot improves. Mode and spirit are set in the TRAINING SUPERVISOR cell.

**How to run:**
1. Ensure **Runtime -> Change runtime type -> T4 GPU**, then restart.
2. Run cells top to bottom.
3. Cell 2 mounts your Drive (click the auth link once). Checkpoints auto-backup to `My Drive/Carnage/checkpoints/`.
4. The TRAINING SUPERVISOR cell launches training and auto-restarts it if it crashes, resuming from the latest checkpoint. The STOP cell ends training cleanly (saves a checkpoint first).

Source: `https://github.com/vfxjamer/Carnage-V1.git`

In [ ]:
# 1. Clone source (self-contained: src + CMakeLists + collision_meshes + GigaLearnCPP thirdparty)
import os, subprocess

ROOT = "/content/Carnage-V1"
REPO = "https://github.com/vfxjamer/Carnage-V1.git"

if not os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
else:
    subprocess.run(["git", "-C", ROOT, "pull"], check=False)

print("ROOT:", ROOT)
print("contents:", sorted(os.listdir(ROOT)))

In [ ]:
# 2. Mount Google Drive EARLY (so checkpoints back up from the start) + restore latest checkpoint.
import os, shutil, glob

if not os.path.isdir("/content/drive"):
    print("mounting Drive (complete the auth popup in the browser)...", flush=True)
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("WARN: drive mount failed:", e, "- backups will retry automatically", flush=True)
else:
    print("drive already mounted", flush=True)

ROOT = "/content/Carnage-V1"
BUILD = os.path.join(ROOT, "build")
DRIVE_CKPT = "/content/drive/MyDrive/Carnage/checkpoints"
LOCAL_CKPT = os.path.join(BUILD, "checkpoints")

os.makedirs(LOCAL_CKPT, exist_ok=True)


def ts_of(d):
    try:
        return int(os.path.basename(d))
    except Exception:
        return -1


if os.path.isdir(DRIVE_CKPT):
    drive_dirs = sorted([d for d in glob.glob(os.path.join(DRIVE_CKPT, "*")) if os.path.isdir(d)], key=ts_of)
    print(f"Drive has {len(drive_dirs)} checkpoint dirs")
    if drive_dirs:
        newest = drive_dirs[-1]
        dest = os.path.join(LOCAL_CKPT, os.path.basename(newest))
        if not os.path.isdir(dest):
            print("restoring latest checkpoint:", os.path.basename(newest))
            shutil.copytree(newest, dest)
else:
    print("no Drive checkpoints yet")

print("local checkpoints:", sorted(os.listdir(LOCAL_CKPT)) if os.path.isdir(LOCAL_CKPT) else [])

In [ ]:
# 3. Install build deps (apt). PyTorch's pip libtorch is used by CMake (no separate libtorch download).
import subprocess, os

APT_PKGS = ["build-essential", "cmake", "git", "libpython3-dev", "pkg-config"]
r = subprocess.run(["apt-get", "update", "-qq"], capture_output=True, text=True)
print("update rc:", r.returncode)
r = subprocess.run(["apt-get", "install", "-y", "-qq"] + APT_PKGS, capture_output=True, text=True)
print("install rc:", r.returncode, (r.stderr or "")[-300:])

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available(), "cuda build:", torch.version.cuda)
if torch.version.cuda is None:
    print("NOTE: torch is CPU-only. Switch to a GPU runtime (Runtime -> Change runtime type -> T4 GPU).")

print(subprocess.run(["cmake", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
print(subprocess.run(["gcc", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

In [ ]:
# 4. Configure Carnage (Release). TORCH_INSTALL_PREFIX points at pip's torch installation.
import os, subprocess

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

import torch
torch_prefix = os.path.dirname(torch.__file__)  # contains share/cmake/Torch
print("TORCH_INSTALL_PREFIX =", torch_prefix)
print("has Torch cmake:", os.path.exists(os.path.join(torch_prefix, "share", "cmake", "Torch")))

configure = [
    "cmake", "-S", ".", "-B", "build",
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DTORCH_INSTALL_PREFIX={torch_prefix}",
]
print(" ".join(configure))
r = subprocess.run(configure, capture_output=True, text=True)
print("configure rc:", r.returncode)
print((r.stdout or "")[-2500:])
print((r.stderr or "")[-1500:])

In [ ]:
# 5. Build (the long step, 10-30 min). Uses all cores.
import os, subprocess

ROOT = "/content/Carnage-V1"
os.chdir(ROOT)

nproc = os.cpu_count() or 2
print(f"Building with -j{nproc} ...")
r = subprocess.run(["cmake", "--build", "build", "-j", str(nproc)], capture_output=True, text=True)
print("build rc:", r.returncode)
print((r.stdout or "")[-3000:])
print((r.stderr or "")[-2000:])
print("---")
exe = os.path.join(ROOT, "build", "Carnage")
print("binary exists:", os.path.exists(exe), exe if os.path.exists(exe) else "")

In [ ]:
# TRAINING SUPERVISOR
# - Auto-selects device: CUDA if available, otherwise CPU (slow but runnable).
# - Starts the 24/7 backup daemon + a live log tailer, then launches training.
# - The tailer streams the binary's real-time output into THIS cell.
# - If the binary crashes, it auto-restarts with backoff, resuming from the latest checkpoint.
# - You should see 'Observation size: 94' (1v1) or 'Observation size: 128' (2v2) after the learner starts.
import os, sys, subprocess, torch, time, threading, shutil, glob

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU detected:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("WARNING: no GPU detected - training on CPU (will be slow)", flush=True)

PLAYERS_PER_TEAM = 2  # 1 = 1v1 (obs 94), 2 = 2v2 (obs 128)
TEAM_SPIRIT = 0.1     # reward sharing with teammates; start low, raise towards 1.0 later

ROOT = "/content/Carnage-V1"
BUILD = os.path.join(ROOT, "build")
LOCAL_CKPT = os.path.join(BUILD, "checkpoints")
DRIVE_CKPT = "/content/drive/MyDrive/Carnage/checkpoints"

# ---- 24/7 backup daemon (guarded: one instance per kernel) ----
def _ts(d):
    try:
        return int(os.path.basename(d))
    except Exception:
        return -1

def _backup_once():
    os.makedirs(DRIVE_CKPT, exist_ok=True)
    local = sorted([d for d in glob.glob(os.path.join(LOCAL_CKPT, "*")) if os.path.isdir(d)], key=_ts)
    drive = set(os.path.basename(d) for d in glob.glob(os.path.join(DRIVE_CKPT, "*")))
    for d in local:
        name = os.path.basename(d)
        if name in drive:
            continue
        tmp = os.path.join(DRIVE_CKPT, name + ".tmp")
        try:
            print(f"[backup] uploading checkpoint {name} ...", flush=True)
            shutil.copytree(d, tmp)
            shutil.move(tmp, os.path.join(DRIVE_CKPT, name))
            print(f"[backup] {name} uploaded", flush=True)
        except Exception as e:
            print("[backup] err:", e, flush=True)
            shutil.rmtree(tmp, ignore_errors=True)

def _backup_loop():
    while True:
        try:
            if os.path.isdir("/content/drive"):
                _backup_once()
            else:
                print("[backup] drive not mounted, retrying in 60s", flush=True)
        except Exception as e:
            print("[backup] loop err:", e, flush=True)
        time.sleep(60)

if "_CARNAGE_BACKUP_DAEMON_" not in globals():
    globals()["_CARNAGE_BACKUP_DAEMON_"] = True
    threading.Thread(target=_backup_loop, daemon=True).start()
    print("[backup] 24/7 daemon started (new checkpoints -> Drive within 60s)", flush=True)

# ---- Live log tailer: streams train.log into this cell ----
if "_CARNAGE_TAILER_" not in globals():
    globals()["_CARNAGE_TAILER_"] = True
    def _tail_loop():
        path = os.path.join(BUILD, "train.log")
        while not os.path.exists(path):
            time.sleep(1)
        with open(path, "r", errors="ignore") as f:
            while True:
                data = f.read()
                if data:
                    sys.stdout.write(data)
                    sys.stdout.flush()
                else:
                    time.sleep(0.5)
    threading.Thread(target=_tail_loop, daemon=True).start()
    print("[tailer] streaming binary output to this cell...", flush=True)

# ---- Training supervisor ----
os.chdir(BUILD)
os.makedirs(LOCAL_CKPT, exist_ok=True)

MESHES_ARG = os.path.join(ROOT, "collision_meshes")
BASE_CMD = ["stdbuf", "-oL", "-eL", "./Carnage", MESHES_ARG, "--device", DEVICE, "--games", "256", "--players-per-team", str(PLAYERS_PER_TEAM), "--team-spirit", str(TEAM_SPIRIT), "--save-dir", "checkpoints"]
print("BASE CMD:", " ".join(BASE_CMD), flush=True)

backoff = 10
attempt = 0
with open("train.log", "a") as log:
    while True:
        attempt += 1
        t0 = time.time()
        print(f"[supervisor] launch #{attempt}: {' '.join(BASE_CMD[:5])} ...", flush=True)
        proc = subprocess.Popen(BASE_CMD, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
        rc = proc.wait()
        elapsed = time.time() - t0
        if rc == 0:
            print("[supervisor] clean exit (rc=0) - stopping supervisor.", flush=True)
            break
        print(f"[supervisor] crashed rc={rc} after {elapsed:.0f}s - restarting in {backoff}s", flush=True)
        time.sleep(backoff)
        if elapsed < 60:
            backoff = min(backoff * 2, 600)
        else:
            backoff = 10
print("supervisor exited.")

In [ ]:
# STATUS: drive mounted? binary running? checkpoints present?
import os, subprocess, glob

print("drive mounted:", os.path.isdir("/content/drive"))
if os.path.isdir("/content/drive"):
    ck = "/content/drive/MyDrive/Carnage/checkpoints"
    print("drive ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(ck + "/*")) if os.path.isdir(ck) else "none yet")

LOCAL = "/content/Carnage-V1/build/checkpoints"
print("local ckpt dirs:", sorted(os.path.basename(d) for d in glob.glob(LOCAL + "/*")) if os.path.isdir(LOCAL) else "none yet")

r = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("Carnage processes:", r.stdout.strip() if r.stdout.strip() else "none")

In [ ]:
# STOP training cleanly: SIGTERM (binary saves a checkpoint then exits), SIGKILL fallback.
import subprocess, time
r = subprocess.run(["pkill", "-TERM", "-f", "Carnage"], capture_output=True, text=True)
print("SIGTERM sent:", r.returncode == 0)
time.sleep(20)
r2 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
alive = r2.stdout.strip()
if alive:
    print("still alive, force killing:", alive)
    subprocess.run(["pkill", "-9", "-f", "Carnage"], capture_output=True)
    time.sleep(3)
r3 = subprocess.run(["pgrep", "-af", "Carnage"], capture_output=True, text=True)
print("remaining:", r3.stdout.strip() if r3.stdout.strip() else "none")